# Unit 5 Real Tools Lab

In this notebook you run a real sentiment model and read its real confidence scores. You do not write any code. You run cells and type text into boxes.

**How to run a cell:** move your mouse over the cell and select the round play button on its left. Wait until the output appears under the cell. Run the cells in order, from top to bottom.

**The model.** It is a public sentiment model named `twitter-roberta-base-sentiment-latest`. Researchers at Cardiff University trained it on about 124 million short social media posts. For any text, it returns three labels (Positive, Neutral, Negative) and a score for each.

Keep your lecture page open in another tab. It tells you what to record after each step.

## Step 0. Load the model

Run this cell first. It takes about one minute. A yellow warning about a "token" or "secrets" may appear. You can ignore it. When the cell prints **Ready**, go to Step 1.

In [ ]:
#@title Step 0. Load the sentiment model (run this first)
import warnings; warnings.filterwarnings("ignore")
from transformers import pipeline, logging as hf_logging
hf_logging.set_verbosity_error()
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
clf = pipeline("sentiment-analysis", model=MODEL, top_k=None)
ORDER = ["positive", "neutral", "negative"]

def shares(text):
    """Return the model's score for each of the three labels."""
    out = clf([text], truncation=True)[0]
    d = {r["label"].lower(): float(r["score"]) for r in out}
    return {k: d.get(k, 0.0) for k in ORDER}

def top(text):
    s = shares(text); k = max(s, key=s.get)
    return k.capitalize(), s[k], s

def show(text):
    lab, sc, s = top(text)
    print("Text:", text); print()
    for k in ORDER:
        print(f"  {k.capitalize():9s} {s[k]:.2f}  " + "#" * round(s[k] * 40))
    print(f"\n  The three scores add up to {sum(s.values()):.2f}.")
    print(f"  The model reports the largest one: {lab}, {sc:.2f}")

fool_log = []
print("Ready. Go to Step 1.")

## Step 1. Read a real confidence score

Type any sentence in the box and run the cell. The model gives each of the three labels a score. The three scores add up to 1. The model reports the label with the largest score. That largest score is the **confidence score**.

Run it at least three times with different sentences. Try one sentence that is clearly happy, one that is clearly angry, and one plain question.

**Record on your lecture page:** one sentence you tried, the label, and the confidence score.

In [ ]:
#@title Step 1. Score one sentence
text = "Thank you, the technician was on time and fixed it." #@param {type:"string"}
show(text)

## Step 2. What did it score: one sentence or the whole message?

A customer left this voicemail for an internet company. The cell scores the **whole voicemail** first. Then it scores **each sentence** separately.

Read both results. Look for the sentence that matters most to the company, and read the label the model gave it.

**Record on your lecture page:** the label for the whole voicemail, and the label for the sentence about the smell.

In [ ]:
#@title Step 2. Score a whole voicemail, then each sentence
voicemail = ["Hi, this is Maria Santos.",
 "A technician installed my new router on Monday.",
 "The router gets very hot and there is a smell like burning plastic near the power adapter.",
 "It is probably nothing.",
 "I do not want to bother anyone.",
 "I also need to move my billing date to the 24th.",
 "Thank you so much, you have all been wonderful."]
lab, sc, _ = top(" ".join(voicemail))
print(f"WHOLE VOICEMAIL: {lab}, {sc:.2f}\n")
print("EACH SENTENCE:")
for s in voicemail:
    lab, sc, _ = top(s)
    print(f"  {lab:9s} {sc:.2f}   {s}")

## Step 3. Ten messages. How many labels are right?

The cell scores ten customer messages. Read each message and decide for yourself what the customer means. Then compare with the model's label.

After the table appears, go to the next cell and type the numbers of the labels you judge **wrong**.

In [ ]:
#@title Step 3a. Score ten customer messages
ten = ["Thank you, the technician was on time and fixed it.",
 "Third outage this week. I want a refund.",
 "Oh wonderful, the bill went up again. Just perfect.",
 "What time does the store close on Sunday?",
 "It would be a little difficult to continue our contract.",
 "The new router is fast. Very happy.",
 "Kill the old account and move everything to the new one, please.",
 "I waited two hours on hold and nobody answered.",
 "My speed test came back negative for packet loss. All good now.",
 "Please send me a copy of my invoice."]
ten_results = []
print(" #  Label      Score  Message")
for i, m in enumerate(ten, 1):
    lab, sc, _ = top(m); ten_results.append((lab, sc))
    print(f"{i:2d}  {lab:9s}  {sc:.2f}   {m}")

In [ ]:
#@title Step 3b. Type the numbers of the labels you judge wrong, with commas between them
wrong_labels = "" #@param {type:"string"}
picked = sorted({int(x) for x in wrong_labels.replace(" ", "").split(",") if x.isdigit() and 1 <= int(x) <= 10})
right = [i for i in range(1, 11) if i not in picked]
print(f"You judged {len(picked)} of 10 labels wrong: {picked if picked else 'none'}")
print(f"You judged {len(right)} of 10 labels right.\n")
avg = lambda idx: sum(ten_results[i-1][1] for i in idx) / len(idx) if idx else 0
if picked: print(f"Average confidence on the labels you judged WRONG: {avg(picked):.2f}")
if right:  print(f"Average confidence on the labels you judged RIGHT: {avg(right):.2f}")
print("\nCompare the two averages. Was the model less confident when it was wrong?")

**Record on your lecture page:** how many of the ten labels you judged right, and the two average confidence scores.

**Why this matters.** The confidence score shows how closely the text matches patterns the model learned. It does not show that the label is right. If the two averages are close, the score did not warn you about the wrong labels.

## Step 4. Try to fool the model

Type a sentence. Choose what you really mean by it. Run the cell. The model is **fooled** when its label does not match your meaning.

Each run adds one line to your log. Fool the model **three times**. Make one of the three a sentence from your own work, your degree program, or your service.

Ideas: sarcasm, a polite refusal, or a word that means something different where you work.

In [ ]:
#@title Step 4. Fool the model (each run adds one try to your log)
sentence = "The test came back positive." #@param {type:"string"}
what_I_really_mean = "Negative" #@param ["Positive", "Neutral", "Negative"]
from_my_own_work = False #@param {type:"boolean"}
lab, sc, _ = top(sentence)
fool_log.append((sentence, what_I_really_mean, lab, sc, lab != what_I_really_mean, from_my_own_work))
print("Fooled  I mean     Model says        Sentence")
for s, mean, lab, sc, fooled, own in fool_log:
    print(f"{'YES' if fooled else 'no ':6s}  {mean:9s}  {lab:9s} {sc:.2f}   {s}{'  (my own work)' if own else ''}")
n = sum(1 for r in fool_log if r[4])
print(f"\nYou have fooled the model {n} time{'s' if n != 1 else ''} in {len(fool_log)} tries.")

**Record on your lecture page:** the sentence from your own work that fooled the model, what you meant, and what the model said.

## Step 5. Is the model reliable?

A reliable system gives the same answer to the same input. This cell scores one message five times.

Then you run the same test on a GenAI tool. Open Gemini in another tab. Paste the prompt below. Send it five times, each time in a **new chat**. Write down the five numbers.

> Score this customer message from 0 to 100 for urgency, where 100 means someone must act today. Reply with the number only. Message: "My internet has been off since morning. My father's medical alert pendant uses it. Sorry to trouble you."

**Record on your lecture page:** the five results from this cell, and the five numbers from Gemini.

In [ ]:
#@title Step 5. Score the same message five times
message = "My internet has been off since morning. My father's medical alert pendant uses it. Sorry to trouble you."
print("Message:", message, "\n")
for i in range(1, 6):
    lab, sc, _ = top(message)
    print(f"  Try {i}: {lab}, {sc:.4f}")
print("\nIs the answer the same every time?")
print("Does the label tell the company how urgent the message is?")

## Step 6. A real test with a threshold

An internet company received 50 customer messages. A supervisor read all of them. **18 are urgent**: someone must call the customer today. **32 are routine.**

The company wants to use the sentiment model to decide who gets a call. It uses the model's **Negative score**, multiplied by 100, as a "call score." A technician calls every customer at or above the threshold.

Run Step 6a once. It scores all 50 messages. Then use Step 6b: move the slider, run the cell, and read the result. The technicians can make **20 calls** a day.

**What you do:**
1. Start at 50. Read the misses.
2. Find the threshold with the fewest misses and 20 calls or fewer.
3. Find the threshold with 0 misses. Count the calls.
4. Read the urgent messages the model scored lowest. Decide why it scored them low.

In [ ]:
#@title Step 6a. Score the 50 messages (run once, about 20 seconds)
URGENT = ["The router is very hot and smells like burning plastic.",
 "There are sparks coming from the wall box!",
 "My internet is down and my father's medical alert pendant needs it. Please help.",
 "The router is quite warm and there is a faint burning smell. No rush, whenever someone is free.",
 "I saw a couple of small sparks at the wall box when it rained. It works fine now. Just letting you know.",
 "Internet has been off since morning. My father's medical alert pendant uses it. Sorry to trouble you.",
 "The adapter makes a buzzing sound and the plug looks a little brown. Thanks.",
 "Your cable is lying across the road after the storm and cars are driving over it.",
 "The pole outside my house is leaning and your line is almost touching the ground.",
 "Smoke came out of the power adapter. I unplugged it. Hope that is okay!",
 "Our clinic has no internet and we cannot open patient records.",
 "The phone line is dead and my mother uses it for her home dialysis alarm.",
 "Thank you for the quick install. One small thing, the box on the wall feels hot when I touch it.",
 "Water is dripping onto your equipment box in the basement. It still works great though.",
 "The technician left an open hole with live wires near the playground.",
 "My security alarm went offline when the internet stopped and I am away for two weeks.",
 "Everything is wonderful with the service. I only noticed a little smoke from the router last night.",
 "The fire station's backup line has been silent since your maintenance yesterday."]
ROUTINE = ["What time does the store close on Sunday?", "Please send me a copy of my invoice.",
 "I would like to upgrade to the faster plan.", "How do I change my Wi-Fi password?",
 "Thank you, the technician was on time and fixed it.", "The new router is fast. Very happy.",
 "Can I move my billing date to the 24th?", "I want to add a second phone line.",
 "Do you offer a discount for students?", "Please update my mailing address.",
 "Great service, thank you so much!", "How do I return the old modem?",
 "Is fiber available on my street yet?", "I love the new app. It is easy to use.",
 "This is ridiculous. My bill went up three dollars and nobody told me.",
 "I HATE your hold music. Worst I have ever heard.",
 "Your website is terrible. I could not find the login button for ten minutes.",
 "I am furious. The technician came at 2:05 and the window was 12 to 2.",
 "Awful. The new logo is ugly and I want the old one back.",
 "Your ads are annoying and I am sick of seeing them.",
 "Worst customer survey ever. Too many questions.",
 "I am very disappointed that the app has no dark mode.",
 "Stop sending me promotional emails. This is unacceptable.",
 "The speed is terrible during the big game every Sunday.",
 "Why is my plan more expensive than my neighbor's? This is unfair.",
 "I waited two hours on hold to ask about a late fee.",
 "My contract ends in March. What are my options?", "Does the modem work with a mesh system?",
 "I will be traveling next month. Can I pause service?", "Please confirm my appointment for Thursday.",
 "What is the data limit on the basic plan?",
 "Kill the old account and move everything to the new one, please."]
scored = [(round(shares(m)["negative"] * 100), True, m) for m in URGENT] + \
         [(round(shares(m)["negative"] * 100), False, m) for m in ROUTINE]
print(f"Scored {len(scored)} messages: {len(URGENT)} urgent and {len(ROUTINE)} routine.")
print("\nThe 18 URGENT messages, from the lowest call score to the highest:")
for sc, u, m in sorted([r for r in scored if r[1]]):
    print(f"  {sc:3d}   {m}")
print("\nThe 5 ROUTINE messages with the highest call scores:")
for sc, u, m in sorted([r for r in scored if not r[1]], reverse=True)[:5]:
    print(f"  {sc:3d}   {m}")

In [ ]:
#@title Step 6b. Move the threshold, then run this cell
threshold = 50 #@param {type:"slider", min:0, max:100, step:5}
caught = [r for r in scored if r[1] and r[0] >= threshold]
missed = [r for r in scored if r[1] and r[0] < threshold]
false_alarms = [r for r in scored if not r[1] and r[0] >= threshold]
calls = len(caught) + len(false_alarms)
print(f"Threshold: {threshold}\n")
print(f"  Urgent, gets a call:           {len(caught):2d}")
print(f"  Urgent, no call (MISSES):      {len(missed):2d}")
print(f"  Routine, gets a call (FALSE ALARMS): {len(false_alarms):2d}")
print(f"  Calls today:                   {calls:2d}   {'OVER the limit of 20' if calls > 20 else 'within the limit of 20'}")
if missed:
    print("\nThe urgent customers who get NO call at this threshold:")
    for sc, u, m in sorted(missed): print(f"  {sc:3d}   {m}")
if false_alarms:
    print("\nThe routine customers who DO get a call:")
    for sc, u, m in sorted(false_alarms, reverse=True): print(f"  {sc:3d}   {m}")

**Record on your lecture page:** your best threshold, its misses, its false alarms, and its number of calls.

**Why no threshold works well.** The model scores how negative the *tone* is. The company needs to know how *urgent* the message is. A calm, polite customer with a burning smell gets a low score. An angry customer who dislikes the hold music gets a high score. The score is reliable, and it is not relevant to this decision. Moving the threshold cannot fix that.

**What a company does about it.** It reads the misses, as you just did. Then it adds a rule (for example, always call when the message contains "burning," "sparks," "smoke," or "medical"), or it uses a model built to score urgency.

## Step 7. Unit 4 follow-up: cross-validation, from the five scores to the final model

This step uses a small real dataset that comes with the scikit-learn library: 178 wines, each with 13 laboratory measurements. The model predicts which of three grape varieties each wine came from.

The cell does what the lecture page describes:
1. It sets aside a test set and does not look at it.
2. It splits the training data into five folds and runs five rounds. Each round builds a model from four folds and tests it on the fifth.
3. It averages the five scores.
4. It builds **one final model** from all of the training data. The five models from the rounds are not kept.
5. It tests the final model **once** on the test set.

**Record on your lecture page:** the five scores, the average, and the final test score.

In [ ]:
#@title Step 7. Five rounds, one final model, one final test
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)
print(f"{len(X)} wines in all. {len(X_train)} for training. {len(X_test)} set aside for the final test.\n")
fold_scores = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=7), X_train, y_train, cv=5)
print("FIVE ROUNDS on the training data:")
for i, s in enumerate(fold_scores, 1):
    print(f"  Round {i}: built from four folds, tested on fold {i}:  {s*100:.0f}% correct")
print(f"\n  Average of the five scores: {fold_scores.mean()*100:.0f}%")
print(f"  Lowest {fold_scores.min()*100:.0f}%, highest {fold_scores.max()*100:.0f}%")
final_model = DecisionTreeClassifier(max_depth=2, random_state=7).fit(X_train, y_train)
print(f"\nFINAL MODEL: built from all {len(X_train)} training wines. The five round models are discarded.")
print(f"FINAL TEST, run once on the {len(X_test)} wines set aside: {final_model.score(X_test, y_test)*100:.0f}% correct")
print("\nIs the final test score close to the average of the five rounds?")

**Why the two numbers should be close.** The average of the five rounds is an estimate of how the model does on wines it has not seen. The final test is a second, separate check of the same thing. When the two agree, you can trust the estimate. When the final test is much lower, something in the test set differs from the training data, and you read those records to find out what.

## You are done

Go back to your lecture page and finish the questions there. To keep a copy of your results, select **File**, then **Print**, and save as a PDF.